<a href="https://colab.research.google.com/github/mistyy12/flyrank-ML/blob/main/notebooks_finished/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mistyy12/flyrank-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [2]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [3]:
REL = "hf://datasets/FlyRank/internship-warehouse"

model_paths = [
    f"{REL}/fact_content_daily_performance/month={month}/*.parquet"
    for month in ["2026-01", "2026-02", "2026-03", "2026-04"]
]

con.read_parquet(model_paths).create_view(
    "model_daily",
    replace=True
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude │ ai_

In [4]:
con.sql("""
    SELECT
        COUNT(*) AS rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS pages

    FROM model_daily
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,first_date,last_date,clients,pages
0,35512033,2026-01-01,2026-04-30,65,380147


In [5]:
page_windows_df = con.sql("""
    WITH daily AS (
        SELECT
            *,
            (
                gsc_data_available IS TRUE
                AND gsc_impressions IS NOT NULL
                AND gsc_clicks IS NOT NULL
            ) AS valid_metrics
        FROM model_daily
        WHERE report_date BETWEEN DATE '2026-01-01'
                              AND DATE '2026-04-30'
    ),

    page_level AS (
        SELECT
            client_hash_id,
            content_hash_id,

            -- January–March features
            COUNT(DISTINCT report_date) FILTER (
                WHERE valid_metrics
                  AND report_date BETWEEN DATE '2026-01-01'
                                      AND DATE '2026-03-31'
            ) AS valid_gsc_days,

            SUM(gsc_impressions) FILTER (
                WHERE valid_metrics
                  AND report_date BETWEEN DATE '2026-01-01'
                                      AND DATE '2026-03-31'
            ) AS impressions,

            SUM(gsc_clicks) FILTER (
                WHERE valid_metrics
                  AND report_date BETWEEN DATE '2026-01-01'
                                      AND DATE '2026-03-31'
            ) AS clicks,

            SUM(gsc_sum_position) FILTER (
                WHERE valid_metrics
                  AND report_date BETWEEN DATE '2026-01-01'
                                      AND DATE '2026-03-31'
                  AND gsc_impressions > 0
                  AND gsc_sum_position IS NOT NULL
                  AND gsc_avg_position > 0
            ) AS position_sum,

            SUM(gsc_impressions) FILTER (
                WHERE valid_metrics
                  AND report_date BETWEEN DATE '2026-01-01'
                                      AND DATE '2026-03-31'
                  AND gsc_impressions > 0
                  AND gsc_sum_position IS NOT NULL
                  AND gsc_avg_position > 0
            ) AS position_impressions,

            -- March reference
            COUNT(DISTINCT report_date) FILTER (
                WHERE valid_metrics
                  AND report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-31'
            ) AS march_valid_days,

            SUM(gsc_impressions) FILTER (
                WHERE valid_metrics
                  AND report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-31'
            ) AS march_impressions,

            -- April outcome
            COUNT(DISTINCT report_date) FILTER (
                WHERE valid_metrics
                  AND report_date BETWEEN DATE '2026-04-01'
                                      AND DATE '2026-04-30'
            ) AS april_valid_days,

            SUM(gsc_impressions) FILTER (
                WHERE valid_metrics
                  AND report_date BETWEEN DATE '2026-04-01'
                                      AND DATE '2026-04-30'
            ) AS april_impressions

        FROM daily
        GROUP BY
          client_hash_id,
          content_hash_id
      )
      SELECT *
      FROM page_level
""").df()

print(f"Rows before eligibility filtering:, {len(page_windows_df):,}")
display(page_windows_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows before eligibility filtering:, 380,147


,client_hash_id,content_hash_id,valid_gsc_days,impressions,clicks,position_sum,position_impressions,march_valid_days,march_impressions,april_valid_days,april_impressions
0,client_62f4a7e64f5e0096,content_65b8a610174a1036,90,6017.0,48.0,28552.0,6017.0,31,1660.0,30,1931.0
1,client_62f4a7e64f5e0096,content_80071808216aef39,90,124246.0,288.0,523563.0,124246.0,31,34253.0,30,39059.0
2,client_62f4a7e64f5e0096,content_9a136f9ba3924c74,90,60387.0,240.0,208223.0,60387.0,31,18068.0,30,18363.0
3,client_62f4a7e64f5e0096,content_200d6a6d1cac69b7,90,7495.0,3.0,6448.0,7348.0,31,3465.0,30,1495.0
4,client_62f4a7e64f5e0096,content_0d440941b1c6807f,90,7660.0,10.0,26662.0,7660.0,31,2172.0,30,1196.0


In [6]:
import numpy as np
import pandas as pd

# January–March CTR
page_windows_df["ctr"] = (
    page_windows_df["clicks"]
    / page_windows_df["impressions"].where(
        page_windows_df["impressions"] > 0
    )
)

# Normalize impressions by valid GSC days.
page_windows_df["march_impressions_per_day"] = (
    page_windows_df["march_impressions"]
    / page_windows_df["march_valid_days"].where(
        page_windows_df["march_valid_days"] > 0
    )
)

page_windows_df["april_impressions_per_day"] = (
    page_windows_df["april_impressions"]
    / page_windows_df["april_valid_days"].where(
        page_windows_df["april_valid_days"] > 0
    )
)

page_windows_df["avg_position"] = (
    page_windows_df["position_sum"]
    / page_windows_df["position_impressions"].where(
        page_windows_df["position_impressions"] > 0
    )
)

print(f"Rows before eligibility filtering:, {len(page_windows_df):,}")
display(page_windows_df.head())


Rows before eligibility filtering:, 380,147


,client_hash_id,content_hash_id,valid_gsc_days,impressions,clicks,position_sum,position_impressions,march_valid_days,march_impressions,april_valid_days,april_impressions,ctr,march_impressions_per_day,april_impressions_per_day,avg_position
0,client_62f4a7e64f5e0096,content_65b8a610174a1036,90,6017.0,48.0,28552.0,6017.0,31,1660.0,30,1931.0,0.007977,53.548387,64.366667,4.745222
1,client_62f4a7e64f5e0096,content_80071808216aef39,90,124246.0,288.0,523563.0,124246.0,31,34253.0,30,39059.0,0.002318,1104.935484,1301.966667,4.213922
2,client_62f4a7e64f5e0096,content_9a136f9ba3924c74,90,60387.0,240.0,208223.0,60387.0,31,18068.0,30,18363.0,0.003974,582.838710,612.100000,3.448143
3,client_62f4a7e64f5e0096,content_200d6a6d1cac69b7,90,7495.0,3.0,6448.0,7348.0,31,3465.0,30,1495.0,0.000400,111.774194,49.833333,0.877518
4,client_62f4a7e64f5e0096,content_0d440941b1c6807f,90,7660.0,10.0,26662.0,7660.0,31,2172.0,30,1196.0,0.001305,70.064516,39.866667,3.480679


In [7]:
MIN_FEATURE_DAYS = 60
MIN_MONTH_DAYS = 20
MIN_IMPRESSIONS = 500
MIN_MARCH_IMPRESSIONS = 100

evaluable = (
    (page_windows_df["valid_gsc_days"] >= MIN_FEATURE_DAYS)
    & (page_windows_df["march_valid_days"] >= MIN_MONTH_DAYS)
    & (page_windows_df["impressions"] >= MIN_IMPRESSIONS)
    & (page_windows_df["march_impressions"] >= MIN_MARCH_IMPRESSIONS)
    & page_windows_df["ctr"].notna()
    & page_windows_df["avg_position"].notna()
)

model_df = page_windows_df.loc[evaluable].copy()

In [8]:
import pandas as pd
import numpy as np

print("Rows:", f"{len(model_df):,}")
print("Columns:", model_df.shape[1])

column_summary = pd.DataFrame({
    "column": model_df.columns,
    "dtype": model_df.dtypes.astype(str).values
})

display(column_summary)
display(model_df.head())

Rows: 57,098
Columns: 15


,column,dtype
0,client_hash_id,object
1,content_hash_id,object
2,valid_gsc_days,int64
3,impressions,float64
4,clicks,float64
5,position_sum,float64
6,position_impressions,float64
7,march_valid_days,int64
8,march_impressions,float64
9,april_valid_days,int64


,client_hash_id,content_hash_id,valid_gsc_days,impressions,clicks,position_sum,position_impressions,march_valid_days,march_impressions,april_valid_days,april_impressions,ctr,march_impressions_per_day,april_impressions_per_day,avg_position
0,client_62f4a7e64f5e0096,content_65b8a610174a1036,90,6017.0,48.0,28552.0,6017.0,31,1660.0,30,1931.0,0.007977,53.548387,64.366667,4.745222
1,client_62f4a7e64f5e0096,content_80071808216aef39,90,124246.0,288.0,523563.0,124246.0,31,34253.0,30,39059.0,0.002318,1104.935484,1301.966667,4.213922
2,client_62f4a7e64f5e0096,content_9a136f9ba3924c74,90,60387.0,240.0,208223.0,60387.0,31,18068.0,30,18363.0,0.003974,582.838710,612.100000,3.448143
3,client_62f4a7e64f5e0096,content_200d6a6d1cac69b7,90,7495.0,3.0,6448.0,7348.0,31,3465.0,30,1495.0,0.000400,111.774194,49.833333,0.877518
4,client_62f4a7e64f5e0096,content_0d440941b1c6807f,90,7660.0,10.0,26662.0,7660.0,31,2172.0,30,1196.0,0.001305,70.064516,39.866667,3.480679


In [9]:
# Duplicate check
duplicate_count = model_df.duplicated(
    subset=["client_hash_id", "content_hash_id"]
).sum()

print("Duplicate client–pages:", duplicate_count)

duplicate_rows = model_df[
    model_df.duplicated(
        subset=["client_hash_id", "content_hash_id"],
        keep=False
    )
].sort_values(
    ["client_hash_id", "content_hash_id"]
)

display(duplicate_rows)

Duplicate client–pages: 0


,client_hash_id,content_hash_id,valid_gsc_days,impressions,clicks,position_sum,position_impressions,march_valid_days,march_impressions,april_valid_days,april_impressions,ctr,march_impressions_per_day,april_impressions_per_day,avg_position


In [10]:
model_df["is_declining_label"] = (
    model_df["april_impressions_per_day"]
    <= 0.80 * model_df["march_impressions_per_day"]
).astype(int)

In [11]:
total_rows = len(model_df)

unique_pairs = (
    model_df[
        ["client_hash_id", "content_hash_id"]
    ]
    .drop_duplicates()
    .shape[0]
)

print("Total rows:", total_rows)
print("Unique client–page pairs:", unique_pairs)
print("Difference:", total_rows - unique_pairs)

Total rows: 57098
Unique client–page pairs: 57098
Difference: 0


In [12]:
KEY_COLS = ["client_hash_id", "content_hash_id"]
FEATURE_COLS = ["valid_gsc_days", "impressions", "ctr", "avg_position"]
TARGET = "is_declining_label"
GROUP = "client_hash_id"

In [13]:
required_columns = (
    KEY_COLS
    + FEATURE_COLS
    + [TARGET]
)

missing_columns = [
    column
    for column in required_columns
    if column not in model_df.columns
]

print("Missing required columns:", missing_columns)

Missing required columns: []


In [14]:
print("Missing required column:", "is_declining_label" not in model_df.columns)

print(model_df["is_declining_label"].value_counts())

print(
    "Positive-label base rate:",
    round(model_df["is_declining_label"].mean(), 3)
)

Missing required column: False
is_declining_label
1    30439
0    26659
Name: count, dtype: int64
Positive-label base rate: 0.533


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


As my task is to rank pages for review, a classifier can learn to predict your binary decline label, then use its predicted probabilities to rank pages.

I use Logistic Regression to estimate the probability that a page meets the April decline label, using January–March GSC features. I rank pages by predicted probability and evaluate the ranking using Precision@20. Logistic Regression provides a simple starting model. I also compare a shallow Decision Tree because it can capture interactions between impressions, CTR, and average position. Both models are compared against my Week-4 rule baseline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Two safeguards:
- Separate time windows: January–March supplies features; April supplies the outcome label.
- Separate clients: pages from a held-out client must not also appear in training.



I use a grouped split by client so that training and test sets contain different clients. This evaluates whether the model can generalize to unseen clients. Features use only January–March measurements, while the label uses April outcomes. I use a fixed random seed and fit preprocessing only on the training set.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [15]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df[FEATURE_COLS],
        model_df[TARGET],
        groups=model_df[GROUP]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()
X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]

X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET]

print("Training pages:", len(train_df))
print("Test pages:", len(test_df))
print("Test declining pages:", int(y_test.sum()))
print("Training clients:", train_df[GROUP].nunique())
print("Test clients:", test_df[GROUP].nunique())
print("Shared clients:", len(
    set(train_df[GROUP]) & set(test_df[GROUP])
))
print("Training labels:", sorted(y_train.unique()))
print("Test labels:", sorted(y_test.unique()))

print("Can calculate Precision@20:", len(test_df) >= 20)
print("Can calculate Precision@50:", len(test_df) >= 50)

Training pages: 35721
Test pages: 21377
Test declining pages: 10573
Training clients: 20
Test clients: 5
Shared clients: 0
Training labels: [np.int64(0), np.int64(1)]
Test labels: [np.int64(0), np.int64(1)]
Can calculate Precision@20: True
Can calculate Precision@50: True


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

1) Prepare x and y
2) Apply week 4 baseline to test_df
3) Train logistic regression
4) Train shallow Decision Tree
5) Rank the same test pages
6) Calculate identical metrices
7) Build comparison table

In [16]:
MIN_IMPRESSIONS = 500
LOW_CTR = 0.01

candidate = ((test_df["impressions"] >= MIN_IMPRESSIONS)
             & (test_df["ctr"] < LOW_CTR)
)
test_df["position_bonus"] = (candidate
             & test_df["avg_position"].between(1, 10))
test_df["baseline_score"] = 0
test_df.loc[candidate, "baseline_score"] = 2
test_df.loc[test_df["position_bonus"], "baseline_score"] = 3


In [17]:
print("Candidate pages:", candidate.sum())

print("\nBaseline score distribution:")
print(
    test_df["baseline_score"]
    .value_counts()
    .sort_index()
)

display(
    test_df[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "ctr",
            "avg_position",
            "position_bonus",
            "baseline_score",
            "is_declining_label"
        ]
    ].head(10)
)

Candidate pages: 21055

Baseline score distribution:
baseline_score
0      322
2     6676
3    14379
Name: count, dtype: int64


,client_hash_id,content_hash_id,impressions,ctr,avg_position,position_bonus,baseline_score,is_declining_label
4419,client_73cda7b4e4f265ea,content_a8ceb031a67c21ed,1469.0,0.002042,6.293956,True,3,1
4420,client_73cda7b4e4f265ea,content_fc079c7770dccaaf,26993.0,0.001519,5.117845,True,3,0
4422,client_73cda7b4e4f265ea,content_3b7f9bad507385b3,960.0,0.002083,12.707627,False,2,0
4423,client_73cda7b4e4f265ea,content_ae37be685ff5d5f1,6002.0,0.001000,6.914791,True,3,1
4424,client_73cda7b4e4f265ea,content_761c9db386b0a983,790.0,0.000000,12.758003,False,2,0
4425,client_73cda7b4e4f265ea,content_4015b9feada77cdf,529.0,0.000000,6.482353,True,3,0
4426,client_73cda7b4e4f265ea,content_b20b16c35f04f2cf,2406.0,0.000831,14.291213,False,2,0
4427,client_73cda7b4e4f265ea,content_96b86f422833e2c7,913.0,0.003286,6.770950,True,3,0
4431,client_73cda7b4e4f265ea,content_1bb74419b3c6997e,12110.0,0.001982,4.255491,True,3,0
4433,client_73cda7b4e4f265ea,content_ddd26b361024ff7f,767.0,0.000000,17.170013,False,2,0


In [18]:
baseline_ranked = test_df.sort_values(
    [
        "baseline_score",
        "impressions",
        "client_hash_id",
        "content_hash_id"
    ],
    ascending=[False, False, True, True]
).copy()

display(
    baseline_ranked[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "ctr",
            "avg_position",
            "baseline_score",
            "is_declining_label"
        ]
    ].head(20)
)

,client_hash_id,content_hash_id,impressions,ctr,avg_position,baseline_score,is_declining_label
48790,client_73cda7b4e4f265ea,content_e241d6415ac9e534,509684.0,0.002462,3.238771,3,0
238491,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,407604.0,0.003960,5.748651,3,0
46715,client_73cda7b4e4f265ea,content_471d9cabce329a66,393421.0,0.002471,4.981224,3,0
90162,client_73cda7b4e4f265ea,content_4d0d79fc12632ef8,388737.0,0.005096,4.604980,3,0
279932,client_73cda7b4e4f265ea,content_6302b8bce0bb84cb,365677.0,0.006580,2.425285,3,0
48372,client_73cda7b4e4f265ea,content_8e1334d6356668e3,362658.0,0.000008,1.216865,3,0
49978,client_73cda7b4e4f265ea,content_cf651123f1085418,360822.0,0.002178,6.005083,3,0
238653,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,353760.0,0.002996,3.693840,3,0
279914,client_73cda7b4e4f265ea,content_b556f0bd87d6fcca,339568.0,0.004762,3.809131,3,0
74696,client_73cda7b4e4f265ea,content_95ff62babbfac9c7,333917.0,0.002060,4.865266,3,0


In [19]:
def precision_at_k(ranked_df, label_col, k):
    if len(ranked_df) < k:
        return float("nan")

    return ranked_df.head(k)[label_col].mean()

In [20]:
baseline_p20 = precision_at_k(
    baseline_ranked,
    TARGET,
    20
)

baseline_p50 = precision_at_k(
    baseline_ranked,
    TARGET,
    50
)

print(f"Baseline Precision@20: {baseline_p20:.3f}")
print(f"Baseline Precision@50: {baseline_p50:.3f}")

Baseline Precision@20: 0.050
Baseline Precision@50: 0.160


In [21]:
base_rate = model_df["is_declining_label"].mean()
print(f"Positive-label base rate: {base_rate:.3f}")

Positive-label base rate: 0.533


In [22]:
test_base_rate = y_test.mean()

print(f"Test base rate: {test_base_rate:.3f}")
print(
    "Expected declining pages in a random top 20:",
    20 * test_base_rate
)

Test base rate: 0.495
Expected declining pages in a random top 20: 9.891939935444636


In [23]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

logistic_model.fit(X_train, y_train)

print("Logistic Regression trained.")

Logistic Regression trained.


In [24]:
test_df["logistic_score"] = (
    logistic_model.predict_proba(X_test)[:, 1]
)

In [25]:
display(
    test_df[
        [
            "client_hash_id",
            "content_hash_id",
            "logistic_score",
            TARGET
        ]
    ].head()
)

print(
    test_df["logistic_score"]
    .describe()
)

,client_hash_id,content_hash_id,logistic_score,is_declining_label
4419,client_73cda7b4e4f265ea,content_a8ceb031a67c21ed,0.619004,1
4420,client_73cda7b4e4f265ea,content_fc079c7770dccaaf,0.597090,0
4422,client_73cda7b4e4f265ea,content_3b7f9bad507385b3,0.575017,0
4423,client_73cda7b4e4f265ea,content_ae37be685ff5d5f1,0.637642,1
4424,client_73cda7b4e4f265ea,content_761c9db386b0a983,0.642224,0


count    21377.000000
mean         0.574041
std          0.078007
min          0.010055
25%          0.538435
50%          0.594423
75%          0.628808
max          0.688050
Name: logistic_score, dtype: float64


In [26]:
from sklearn.tree import DecisionTreeClassifier

tree_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        DecisionTreeClassifier(
            max_depth=3,
            min_samples_leaf=20,
            random_state=42
        )
    )
])

tree_model.fit(X_train, y_train)

print("Shallow Decision Tree trained.")

Shallow Decision Tree trained.


In [29]:
test_df["tree_score"] = (
    tree_model.predict_proba(X_test)[:, 1]
)

In [30]:
display(
    test_df[
        [
            "client_hash_id",
            "content_hash_id",
            "tree_score",
            TARGET
        ]
    ].head()
)

print(
    test_df["tree_score"]
    .describe()
)

,client_hash_id,content_hash_id,tree_score,is_declining_label
4419,client_73cda7b4e4f265ea,content_a8ceb031a67c21ed,0.756324,1
4420,client_73cda7b4e4f265ea,content_fc079c7770dccaaf,0.756324,0
4422,client_73cda7b4e4f265ea,content_3b7f9bad507385b3,0.445944,0
4423,client_73cda7b4e4f265ea,content_ae37be685ff5d5f1,0.756324,1
4424,client_73cda7b4e4f265ea,content_761c9db386b0a983,0.445944,0


count    21377.000000
mean         0.620881
std          0.127840
min          0.000000
25%          0.532342
50%          0.639643
75%          0.756324
max          0.756324
Name: tree_score, dtype: float64


In [31]:
score_check = test_df[
    [
        "baseline_score",
        "logistic_score",
        "tree_score"
    ]
].describe().T

display(score_check)

,count,mean,std,min,25%,50%,75%,max
baseline_score,21377.0,2.642513,0.565758,0.000000,2.000000,3.000000,3.000000,3.000000
logistic_score,21377.0,0.574041,0.078007,0.010055,0.538435,0.594423,0.628808,0.688050
tree_score,21377.0,0.620881,0.127840,0.000000,0.532342,0.639643,0.756324,0.756324


In [32]:
logistic_ranked = test_df.sort_values(
    [
        "logistic_score",
        "impressions",
        "client_hash_id",
        "content_hash_id"
    ],
    ascending=[False, False, True, True]
).copy()

tree_ranked = test_df.sort_values(
    [
        "tree_score",
        "impressions",
        "client_hash_id",
        "content_hash_id"
    ],
    ascending=[False, False, True, True]
).copy()

In [33]:
logistic_p20 = precision_at_k(
    logistic_ranked,
    TARGET,
    20
)

logistic_p50 = precision_at_k(
    logistic_ranked,
    TARGET,
    50
)

tree_p20 = precision_at_k(
    tree_ranked,
    TARGET,
    20
)

tree_p50 = precision_at_k(
    tree_ranked,
    TARGET,
    50
)

print(f"Logistic Precision@20: {logistic_p20:.3f}")
print(f"Logistic Precision@50: {logistic_p50:.3f}")
print(f"Tree Precision@20: {tree_p20:.3f}")
print(f"Tree Precision@50: {tree_p50:.3f}")

Logistic Precision@20: 0.800
Logistic Precision@50: 0.820
Tree Precision@20: 0.400
Tree Precision@50: 0.360


In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

comparison_df = pd.DataFrame({
    "Method": [
        "Test-set base rate",
        "Week-4 rule baseline",
        "Logistic Regression",
        "Shallow Decision Tree"
    ],
    "Precision@20": [
        test_base_rate,
        baseline_p20,
        logistic_p20,
        tree_p20
    ],
    "Precision@50": [
        test_base_rate,
        baseline_p50,
        logistic_p50,
        tree_p50
    ]
})

display(comparison_df.round(3))


,Method,Precision@20,Precision@50
0,Test-set base rate,0.495,0.495
1,Week-4 rule baseline,0.050,0.160
2,Logistic Regression,0.800,0.820
3,Shallow Decision Tree,0.400,0.360


In [35]:
print("Features used:", FEATURE_COLS)


print(
    "Logistic scores outside 0–1:",
    (
        (test_df["logistic_score"] < 0)
        | (test_df["logistic_score"] > 1)
    ).sum()
)

print(
    "Logistic top-20 positives:",
    logistic_ranked.head(20)[TARGET].sum()
)

print(
    "Logistic top-50 positives:",
    logistic_ranked.head(50)[TARGET].sum()
)

Features used: ['valid_gsc_days', 'impressions', 'ctr', 'avg_position']
Logistic scores outside 0–1: 0
Logistic top-20 positives: 16
Logistic top-50 positives: 41


In [36]:
LEAKAGE_COLS = {
    "april_valid_days",
    "april_impressions",
    "april_daily_impressions",
    "impression_change_pct",
    "is_declining_label"
}

print(
    "Leaking features:",
    set(FEATURE_COLS) & LEAKAGE_COLS
)

Leaking features: set()


In [37]:
logistic_top20 = logistic_ranked.head(20).copy()

logistic_top20["rank"] = range(
    1,
    len(logistic_top20) + 1
)

logistic_top20["ctr_pct"] = (
    logistic_top20["ctr"] * 100
)

logistic_top20["april_change_pct"] = (
    (
        logistic_top20["april_impressions_per_day"]
        / logistic_top20["march_impressions_per_day"]
    ) - 1
) * 100

In [38]:
display(
    logistic_top20[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "valid_gsc_days",
            "impressions",
            "ctr_pct",
            "avg_position",
            "logistic_score",
            "march_impressions_per_day",
            "april_impressions_per_day",
            "april_change_pct",
            "is_declining_label"
        ]
    ].round(3)
)

,rank,client_hash_id,content_hash_id,valid_gsc_days,impressions,ctr_pct,avg_position,logistic_score,march_impressions_per_day,april_impressions_per_day,april_change_pct,is_declining_label
279123,1,client_73cda7b4e4f265ea,content_38cf94983ceed0d6,90,1361.0,0.0,1.098,0.688,11.871,8.333,-29.801,1
88595,2,client_73cda7b4e4f265ea,content_c3ecf7aa4f3ffd0f,90,2013.0,0.0,0.828,0.688,14.355,7.185,-49.946,1
279106,3,client_73cda7b4e4f265ea,content_0894fdf97cc8e32c,90,903.0,0.0,1.402,0.688,9.129,4.310,-52.784,1
88499,4,client_73cda7b4e4f265ea,content_221bc921d7feb436,90,821.0,0.0,1.596,0.687,7.032,7.724,9.839,0
278980,5,client_73cda7b4e4f265ea,content_b7d396329f1e4c69,90,1810.0,0.0,1.262,0.687,24.097,10.200,-57.671,1
194962,6,client_fef1a8f436438636,content_133021cfe89c98db,90,1340.0,0.0,1.493,0.687,19.258,5.852,-69.613,1
266592,7,client_73cda7b4e4f265ea,content_146998d49b077979,90,1832.0,0.0,1.295,0.687,6.742,3.593,-46.713,1
88426,8,client_73cda7b4e4f265ea,content_ca9141cc4b189a4c,90,1797.0,0.0,1.358,0.687,27.710,13.586,-50.969,1
88626,9,client_73cda7b4e4f265ea,content_94ca3f182cdff8a5,90,1353.0,0.0,1.591,0.687,15.484,11.733,-24.222,1
88611,10,client_73cda7b4e4f265ea,content_b4cb066050192ed2,90,1036.0,0.0,1.788,0.686,14.677,10.400,-29.143,1


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Logistic Regression achieved the highest ranking precision among the three methods, with Precision@20 of 0.800 and Precision@50 of 0.687, compared with a test-set base rate of 0.415. It was therefore the strongest method on this client-grouped split. However, its top 20 still included four false positives.

No. 4, where its page receive a decline score of 0.704 where its daily impressions increase by approximately 9.8%, from 7.03 in March to 7.72 in April. This cause false positive.

No. 17, where its page receive a decline score of 0.685, but daily impression increased by approximately 4.67%, from 5.77 to 6.04. This shows a ranking error, where false positives happen.

Logistic Regression combines the input features using learned coefficients. Holding other features fixed, a positive coefficient means higher feature values increase predicted decline risk, while a negative coefficient means they decrease it. Because the features were standardized, larger absolute coefficients indicate a larger effect on the model’s log-odds per standard-deviation increase.

In [ ]:
top20_false_positives = logistic_ranked.head(20).loc[
    lambda df: df["is_declining_label"] == 0
].copy()

display(
    top20_false_positives[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "ctr",
            "avg_position",
            "logistic_score",
            "march_impressions_per_day",
            "april_impressions_per_day",
            "is_declining_label"
        ]
    ]
)

,client_hash_id,content_hash_id,impressions,ctr,avg_position,logistic_score,march_impressions_per_day,april_impressions_per_day,is_declining_label
3,client_73cda7b4e4f265ea,content_221bc921d7feb436,821.0,0.0,1.595620,0.704500,7.032258,7.724138,0
10,client_73cda7b4e4f265ea,content_1ea90b26ddfc77d1,1265.0,0.0,1.895954,0.702844,16.129032,17.900000,0
16,client_73cda7b4e4f265ea,content_42dba87391e1830f,631.0,0.0,2.477723,0.701929,7.935484,16.692308,0
17,client_73cda7b4e4f265ea,content_9741212827b86fa1,1314.0,0.0,2.168529,0.701883,18.354839,31.900000,0


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
logistic_ranked = test_df.sort_values(
    [
        "logistic_score",
        "impressions",
        "client_hash_id",
        "content_hash_id"
    ],
    ascending=[False, False, True, True]
).reset_index(drop=True)

logistic_ranked["logistic_rank"] = (
    logistic_ranked.index + 1
)

logistic_ranked["actual_change_pct"] = (
    (
        logistic_ranked["april_impressions_per_day"]
        / logistic_ranked["march_impressions_per_day"]
    ) - 1
) * 100


In [ ]:
false_positives = logistic_ranked.loc[
    (logistic_ranked["logistic_rank"] <= 20)
    & (logistic_ranked[TARGET] == 0)
].copy()

In [ ]:
missed_positives = logistic_ranked.loc[
    (logistic_ranked["logistic_rank"] > 20)
    & (logistic_ranked[TARGET] == 1)
].copy()

missed_positives = missed_positives.sort_values(
    "logistic_score",
    ascending=True
)

In [ ]:
print(
    "Top-20 correct declining pages:",
    logistic_ranked.head(20)[TARGET].sum()
)

print(
    "Top-20 false positives:",
    len(false_positives)
)

print(
    "Declining pages outside top 20:",
    len(missed_positives)
)

Top-20 correct declining pages: 16
Top-20 false positives: 4
Declining pages outside top 20: 6514


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.